# Upload ChildWhisper Data: Google Drive → Kaggle Dataset

Run this notebook in **Google Colab** to transfer data from Google Drive to Kaggle.
This is fast because data moves within Google's network (no local download needed).

**Prerequisites:**
1. Data in Google Drive (audio zips + metadata JSONL)
2. Kaggle API token (`kaggle.json`) — get from https://www.kaggle.com/settings/account

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure Paths

Update `GDRIVE_FOLDER` to point to your data folder in Google Drive.

In [ ]:
import os
from pathlib import Path

# ── EDIT THIS: path to your data folder in Google Drive ──
GDRIVE_FOLDER = "/content/drive/MyDrive/childwhisper/data"
# If your files are at the root of the shared folder, try:
# GDRIVE_FOLDER = "/content/drive/MyDrive"

# Kaggle dataset config
KAGGLE_USERNAME = "nishantgaurav23"
KAGGLE_DATASET_SLUG = f"{KAGGLE_USERNAME}/pasketti-audio"

# Staging directory (Colab local disk — fast SSD)
STAGING_DIR = Path("/content/kaggle_dataset")
STAGING_DIR.mkdir(parents=True, exist_ok=True)

# Check what's in the Drive folder
gdrive = Path(GDRIVE_FOLDER)
if gdrive.exists():
    print("Files in Google Drive folder:")
    for f in sorted(gdrive.iterdir()):
        size_mb = f.stat().st_size / 1024 / 1024 if f.is_file() else 0
        print(f"  {f.name:40s} {size_mb:>8.1f} MB" if f.is_file() else f"  {f.name:40s} [DIR]")
else:
    print(f"ERROR: Folder not found: {GDRIVE_FOLDER}")
    print("Browse /content/drive/MyDrive/ to find your data folder")
    !ls /content/drive/MyDrive/

## 3. Setup Kaggle API

In [ ]:
!pip install -q kaggle

# Option A: Upload kaggle.json from your computer
# from google.colab import files
# uploaded = files.upload()  # Upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Option B: If kaggle.json is in your Google Drive
KAGGLE_JSON = "/content/drive/MyDrive/kaggle.json"
if Path(KAGGLE_JSON).exists():
    !mkdir -p ~/.kaggle && cp {KAGGLE_JSON} ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle credentials loaded from Google Drive")
else:
    print(f"kaggle.json not found at {KAGGLE_JSON}")
    print("Either:")
    print("  1. Upload kaggle.json to Google Drive and update KAGGLE_JSON path above")
    print("  2. Uncomment Option A above to upload directly")
    print("  3. Set env vars manually:")
    print('     os.environ["KAGGLE_USERNAME"] = "your_username"')
    print('     os.environ["KAGGLE_KEY"] = "your_api_key"')

# Verify
!kaggle datasets list --mine 2>&1 | head -3

## 4. Extract & Organize Data

Copy files from Drive to local staging, extract zips, and organize into the expected structure:
```
kaggle_dataset/
├── audio/               # All .flac files
├── train_word_transcripts.jsonl
└── train_phon_transcripts.jsonl  (if available)
```

In [ ]:
import shutil
import zipfile
import tarfile
import time

gdrive = Path(GDRIVE_FOLDER)

# Step 1: Copy metadata files (small, fast)
print("=== Copying metadata files ===")
for jsonl in gdrive.glob("*.jsonl"):
    dest = STAGING_DIR / jsonl.name
    if not dest.exists():
        shutil.copy2(jsonl, dest)
        print(f"  Copied {jsonl.name}")
    else:
        print(f"  Skipping {jsonl.name} (already exists)")

# Step 2: Extract audio zips
print("\n=== Extracting audio archives ===")
audio_dir = STAGING_DIR / "audio"
audio_dir.mkdir(exist_ok=True)

for archive in sorted(gdrive.glob("audio*.zip")):
    print(f"  Extracting {archive.name} ({archive.stat().st_size / 1024**3:.1f} GB)...")
    t0 = time.time()
    with zipfile.ZipFile(archive, 'r') as zf:
        zf.extractall(STAGING_DIR)
    print(f"    Done in {time.time() - t0:.0f}s")

# Step 3: Extract noise data if present
noise_dir = STAGING_DIR / "noise"
for archive in sorted(gdrive.glob("noise*.zip")):
    print(f"  Extracting {archive.name}...")
    t0 = time.time()
    with zipfile.ZipFile(archive, 'r') as zf:
        zf.extractall(STAGING_DIR)
    print(f"    Done in {time.time() - t0:.0f}s")

for archive in gdrive.glob("musan*.tar.gz"):
    print(f"  Extracting {archive.name}...")
    t0 = time.time()
    with tarfile.open(archive, 'r:gz') as tf:
        tf.extractall(STAGING_DIR)
    print(f"    Done in {time.time() - t0:.0f}s")

# Summary
print("\n=== Staging directory contents ===")
!du -sh {STAGING_DIR}/*

In [ ]:
# Verify: count audio files and check metadata
import json

audio_dir = STAGING_DIR / "audio"
flac_files = list(audio_dir.rglob("*.flac"))
wav_files = list(audio_dir.rglob("*.wav"))
print(f"Audio files: {len(flac_files)} FLAC, {len(wav_files)} WAV")

meta_path = STAGING_DIR / "train_word_transcripts.jsonl"
if meta_path.exists():
    with open(meta_path) as f:
        entries = [json.loads(line) for line in f if line.strip()]
    print(f"Metadata entries: {len(entries)}")
    if entries:
        print(f"Sample entry keys: {list(entries[0].keys())}")
        # Check how many audio files match
        matched = sum(1 for e in entries if (audio_dir / e.get('audio_path', '')).exists())
        print(f"Audio files matched to metadata: {matched}/{len(entries)}")
else:
    print("WARNING: train_word_transcripts.jsonl not found!")

## 5. Upload to Kaggle as Dataset

This creates a private Kaggle dataset. Once uploaded, your training notebooks
access it at `/kaggle/input/pasketti-audio/`.

In [ ]:
# Create Kaggle dataset metadata
import json

dataset_meta = {
    "title": "Pasketti Children ASR Audio",
    "id": KAGGLE_DATASET_SLUG,
    "licenses": [{"name": "CC-BY-4.0"}]
}

meta_file = STAGING_DIR / "dataset-metadata.json"
meta_file.write_text(json.dumps(dataset_meta, indent=2))
print(f"Dataset metadata written to {meta_file}")
print(json.dumps(dataset_meta, indent=2))

In [ ]:
%%time
# Upload to Kaggle (this can take 10-30 min for large datasets)
# First attempt: create new dataset
# If it already exists, update with a new version

import subprocess

result = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(STAGING_DIR), "--dir-mode", "zip"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("Dataset created successfully!")
    print(result.stdout)
elif "already exists" in result.stderr.lower() or "already exists" in result.stdout.lower():
    print("Dataset already exists, creating new version...")
    result2 = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(STAGING_DIR),
         "-m", "Updated from Google Drive", "--dir-mode", "zip"],
        capture_output=True, text=True
    )
    print(result2.stdout)
    if result2.returncode != 0:
        print(f"Error: {result2.stderr}")
else:
    print(f"Error creating dataset: {result.stderr}")
    print(f"stdout: {result.stdout}")

In [ ]:
# Verify the dataset exists on Kaggle
!kaggle datasets status {KAGGLE_DATASET_SLUG}

## 6. (Optional) Upload Noise Data as Separate Dataset

If you have noise data (MUSAN/RealClass), upload as a separate Kaggle dataset
so the augmented training notebook can use it.

In [ ]:
# Check if noise data was extracted
noise_dirs = [
    STAGING_DIR / "noise",
    STAGING_DIR / "musan",
]

has_noise = any(d.exists() for d in noise_dirs)
if has_noise:
    print("Noise data found. To upload as a separate Kaggle dataset:")
    print("  1. Create a new staging dir with just the noise files")
    print("  2. Upload with slug: nishantgaurav23/childwhisper-noise")
    for d in noise_dirs:
        if d.exists():
            n_files = len(list(d.rglob("*")))
            print(f"  Found: {d.name}/ ({n_files} files)")
else:
    print("No noise data found. Skipping.")

## Done!

Your data is now available as a Kaggle dataset. In your training notebooks:

```python
from src.kaggle_utils import get_paths

paths = get_paths(dataset_slug="pasketti-audio")
# paths["audio_dir"]      → /kaggle/input/pasketti-audio/audio
# paths["metadata_path"]  → /kaggle/input/pasketti-audio/train_word_transcripts.jsonl
```

Add `nishantgaurav23/pasketti-audio` to your kernel's `dataset_sources` in `kernel-metadata.json`.